# Malliavin Greeks: Theory and Variance Comparison

This notebook demonstrates the `malliavin-greeks` library: an implementation of Malliavin calculus methods for computing sensitivities (Greeks) of derivative securities via Monte Carlo.

## Key idea

Rather than perturbing the initial condition and re-simulating (finite differences), Malliavin calculus derives **weight functions** $\pi$ such that
$$
\text{Greek} = e^{-rT}\,\mathbb{E}\bigl[f(S_T)\,\pi\bigr]
$$
from a **single** set of simulation paths.  The weights are derived by integration by parts on Wiener space (the Bismut–Elworthy–Li formula) and work for **discontinuous** payoffs (digital options, barriers) without modification.

## Outline
1. Setup and model parameters
2. Delta: Malliavin vs. finite differences vs. pathwise vs. likelihood-ratio
3. Gamma: Malliavin vs. FD — variance comparison
4. Vega: Malliavin vs. pathwise
5. Digital options — the discontinuous payoff case
6. Asian option delta — path-dependent payoffs
7. Variance reduction: control variates and importance sampling
8. Heston model Greeks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from malliavin_greeks.models import GBMParams, HestonParams
from malliavin_greeks.payoffs import european_call, european_put, digital_call, asian_call
from malliavin_greeks.estimators import (
    malliavin_delta, malliavin_gamma, malliavin_vega, malliavin_rho,
    fd_delta_central, fd_gamma_central, fd_vega_central,
    pathwise_delta, pathwise_vega, lr_delta, lr_gamma,
    pathwise_delta_asian,
)
from malliavin_greeks.variance import cv_delta_call, is_delta_digital, stratified_delta
from malliavin_greeks.benchmarks import bs_greeks, benchmark_delta, benchmark_gamma, benchmark_vega, benchmark_digital_delta, print_benchmark_table

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Library loaded.')

## 1. Model Parameters

We use the standard GBM risk-neutral model throughout:
$$
dS_t = (r - q)\,S_t\,dt + \sigma\,S_t\,dW_t, \qquad S_0 = 100.
$$

In [ ]:
params = GBMParams(S0=100.0, r=0.05, q=0.0, sigma=0.20, T=1.0)
K_ATM = 100.0
K_OTM = 120.0
K_ITM = 80.0

# Black-Scholes reference values
bs = bs_greeks(params, K_ATM)
print('Black-Scholes ATM call Greeks:')
for k, v in bs.items():
    print(f'  {k:8s} = {v:.6f}')

## 2. Delta Comparison

The **Malliavin delta weight** under GBM is derived from the score function of the log-normal density:
$$
\pi_\Delta = \frac{W_T}{\sigma\,S_0\,T}
$$
where $W_T = \sigma\sqrt{dt}\sum_i Z_i$ is the terminal Brownian motion.  This coincides with the likelihood-ratio (LR) score function for terminal payoffs.

In [ ]:
N = 200_000
seed = 42

results_delta = benchmark_delta(params, K_ATM, n_paths=N, seed=seed)
print_benchmark_table(results_delta, 'Delta: ATM European Call (N=200,000)')

In [ ]:
# Convergence plot: std error vs sqrt(N)
n_vals = [5_000, 10_000, 25_000, 50_000, 100_000, 200_000]
truth_delta = bs_greeks(params, K_ATM)['delta']

se_mall, se_fd, se_pw = [], [], []
for n in n_vals:
    r_m = malliavin_delta(params, lambda S: european_call(S, K_ATM), n, 1, np.random.default_rng(seed))
    r_fd = fd_delta_central(params, lambda S: european_call(S, K_ATM), n, 1, 1.0, np.random.default_rng(seed))
    r_pw = pathwise_delta(params, lambda S: (S[:, -1] > K_ATM).astype(float), n, 1, np.random.default_rng(seed))
    se_mall.append(r_m.std_err)
    se_fd.append(r_fd.std_err)
    se_pw.append(r_pw.std_err)

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(n_vals, se_mall, 'b-o', label='Malliavin / LR')
ax.loglog(n_vals, se_fd,   'r-s', label='FD Central')
ax.loglog(n_vals, se_pw,   'g-^', label='Pathwise (IPA)')
# Reference O(1/sqrt(N)) line
ref = se_mall[0] * np.sqrt(n_vals[0] / np.array(n_vals))
ax.loglog(n_vals, ref, 'k--', alpha=0.5, label=r'$O(N^{-1/2})$')
ax.set_xlabel('Number of paths N')
ax.set_ylabel('Standard error')
ax.set_title('Delta convergence: std error vs N')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'All methods converge at O(1/sqrt(N)), but pathwise has lowest variance for smooth payoffs.')

## 3. Gamma Comparison

The **Malliavin gamma weight** is obtained by differentiating the delta score twice:
$$
\pi_\Gamma = \frac{W_T(W_T - \sigma T) - T}{\sigma^2 T^2 S_0^2}
$$
Finite differences for gamma require **three** price simulations; Malliavin needs **one**.
For discontinuous payoffs (digital, barrier) where finite-difference gamma is extremely noisy, Malliavin is especially advantageous.

In [ ]:
results_gamma = benchmark_gamma(params, K_ATM, n_paths=N, seed=seed)
print_benchmark_table(results_gamma, 'Gamma: ATM European Call (N=200,000)')

## 4. Vega Comparison

The **Malliavin vega weight** (sensitivity to $\sigma$) follows from the score function of the log-normal density w.r.t. $\sigma$:
$$
\pi_\nu = \frac{W_T^2 - T}{\sigma T} - W_T
$$
Note that $\mathbb{E}[\pi_\nu] = 0$ (score functions are mean-zero), so no bias is introduced.

In [ ]:
results_vega = benchmark_vega(params, K_ATM, n_paths=N, seed=seed)
print_benchmark_table(results_vega, 'Vega: ATM European Call (N=200,000)')

## 5. Digital Options — Discontinuous Payoffs

This is the **key use case** for Malliavin calculus.  The digital call payoff $f(S_T) = \mathbf{1}_{S_T > K}$ is discontinuous.

- **Finite differences**: bump $S_0 \pm h$ and revalue.  For $h$ large: high bias.  For $h$ small: high variance (two independent simulations of a discontinuous function).  **No good bump size**.
- **Pathwise (IPA)**: $f'(S_T) = \delta(S_T - K)$ — not defined.
- **Malliavin / LR**: the weight $\pi_\Delta$ is exactly the same as for a call; **no modification needed**.

In [ ]:
results_digital = benchmark_digital_delta(params, K_ATM, n_paths=N, seed=seed)
print_benchmark_table(results_digital, 'Delta: ATM Digital Call (N=200,000)')

In [ ]:
# Show FD bias-variance tradeoff for digital delta
from scipy.stats import norm

p = params
d2 = (np.log(p.S0/K_ATM) + (p.r - 0.5*p.sigma**2)*p.T) / (p.sigma*np.sqrt(p.T))
truth_digital = np.exp(-p.r*p.T) * norm.pdf(d2) / (p.S0*p.sigma*np.sqrt(p.T))

h_vals = np.logspace(-1, 1.5, 20)
fd_means, fd_ses = [], []
n_fd = 100_000
for h in h_vals:
    r = fd_delta_central(params, lambda S: digital_call(S, K_ATM), n_fd, 1, h, np.random.default_rng(seed))
    fd_means.append(r.greek)
    fd_ses.append(r.std_err)

fd_means = np.array(fd_means)
fd_ses = np.array(fd_ses)

# Malliavin reference
r_mall = malliavin_delta(params, lambda S: digital_call(S, K_ATM), n_fd, 1, np.random.default_rng(seed))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.semilogx(h_vals, fd_means, 'r-o', label='FD Central estimate')
ax1.axhline(truth_digital, color='k', linestyle='--', label='Truth')
ax1.axhline(r_mall.greek, color='b', linestyle='-', label='Malliavin estimate')
ax1.fill_between(h_vals, fd_means - 2*fd_ses, fd_means + 2*fd_ses, alpha=0.2, color='red')
ax1.set_xlabel('Bump size h')
ax1.set_ylabel('Delta estimate')
ax1.set_title('FD bias-variance tradeoff: Digital Delta')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.semilogx(h_vals, fd_ses, 'r-o', label='FD std error')
ax2.axhline(r_mall.std_err, color='b', linestyle='-', label='Malliavin std error')
ax2.set_xlabel('Bump size h')
ax2.set_ylabel('Standard error')
ax2.set_title('Standard error vs bump size')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('Malliavin: unbiased for any h (no h needed). FD: bias for large h, variance for small h.')

## 6. Asian Option Delta

For the arithmetic-average Asian call $f(A) = (A - K)^+$ where $A = \frac{1}{n}\sum_{i=1}^n S_{t_i}$:

- The average $A$ is **smooth** in $S_0$ (since $A = S_0 \cdot A_{\text{normalized}}$), so the **pathwise IPA** estimator applies:
$$
\Delta_\text{Asian} = e^{-rT}\,\mathbb{E}\bigl[\mathbf{1}_{A > K}\cdot A/S_0\bigr]
$$
- For **digital Asian** options ($f(A) = \mathbf{1}_{A > K}$), pathwise fails and Malliavin IBP is required.

In [ ]:
n_steps = 52  # weekly monitoring
n_asian = 200_000

r_pw_asian = pathwise_delta_asian(
    params, lambda S, A: (A > K_ATM).astype(float),
    n_asian, n_steps, np.random.default_rng(seed)
)
r_fd_asian = fd_delta_central(
    params, lambda S: asian_call(S, K_ATM),
    n_asian, n_steps, 0.5, np.random.default_rng(seed)
)

print(f'Pathwise Asian Delta: {r_pw_asian.greek:.5f}  ±{r_pw_asian.std_err:.5f}')
print(f'FD Central   Asian Delta: {r_fd_asian.greek:.5f}  ±{r_fd_asian.std_err:.5f}')
print(f'Pathwise variance: {r_pw_asian.variance:.4f}')
print(f'FD variance:       {r_fd_asian.variance:.4f}')
print(f'Variance ratio (FD/PW): {r_fd_asian.variance/r_pw_asian.variance:.2f}x  (pathwise {r_fd_asian.variance/r_pw_asian.variance:.1f}x more efficient than FD)')

## 7. Variance Reduction

Three variance-reduction techniques specifically designed for Malliavin estimators:

1. **Antithetic variates**: simulate $Z$ and $-Z$; already enabled by `antithetic=True`.
2. **Control variates**: use the discounted payoff (with known mean = BS price) as control.
3. **Importance sampling**: shift the drift to concentrate paths near the payoff region.

In [ ]:
n_vr = 100_000

r_plain = malliavin_delta(params, lambda S: european_call(S, K_ATM), n_vr, 1, np.random.default_rng(seed), antithetic=False)
r_anti  = malliavin_delta(params, lambda S: european_call(S, K_ATM), n_vr, 1, np.random.default_rng(seed), antithetic=True)
r_cv    = cv_delta_call(params, K_ATM, n_vr, 1, np.random.default_rng(seed), antithetic=True)
r_strat = stratified_delta(params, lambda S: european_call(S, K_ATM), n_vr, n_strata=20, rng=np.random.default_rng(seed))

truth = bs_greeks(params, K_ATM)['delta']
methods = ['Plain Malliavin', 'Antithetic', 'Anti + Control Variate', 'Stratified (20)']
results_vr = [r_plain, r_anti, r_cv, r_strat]

print(f'{"Method":<25} {"Estimate":>10} {"Std Err":>10} {"Variance":>12} {"Var Reduction":>15}')
print('-'*75)
base_var = r_plain.variance
for m, r in zip(methods, results_vr):
    print(f'{m:<25} {r.greek:>10.5f} {r.std_err:>10.5f} {r.variance:>12.4e} {base_var/r.variance:>14.1f}x')
print(f'Truth: {truth:.5f}')

## 8. Heston Model Greeks

Under the Heston stochastic volatility model:
$$
dS_t = rS_t\,dt + \sqrt{V_t}\,S_t\,dW^S_t, \qquad dV_t = \kappa(\theta - V_t)\,dt + \xi\sqrt{V_t}\,dW^V_t
$$
the Malliavin delta weight uses the BEL formula with $u(t) = 1/T$:
$$
\pi_\Delta^\text{Heston} = \frac{1}{S_0 T}\int_0^T \frac{1}{\sqrt{V_t}}\,dW^S_t
$$
This requires paths of both $S$ and $V$, but remains a single-simulation estimator.

In [ ]:
from malliavin_greeks.models import simulate_heston
from malliavin_greeks import weights as W

h_params = HestonParams(
    S0=100.0, V0=0.04, r=0.05, q=0.0,
    kappa=2.0, theta=0.04, xi=0.30, rho=-0.70, T=1.0
)

n_heston = 200_000
n_steps_h = 100

S, V, Z1, Z2 = simulate_heston(h_params, n_heston, n_steps_h, np.random.default_rng(seed))
pi_delta = W.weight_delta_heston(h_params, S, V, Z1, Z2)
disc = np.exp(-h_params.r * h_params.T)

payoff = european_call(S, K_ATM)
delta_samples = disc * payoff * pi_delta

delta_est = delta_samples.mean()
delta_se  = delta_samples.std() / np.sqrt(n_heston)

# Compare to FD (bump S0)
from malliavin_greeks.models import GBMParams
h_up = HestonParams(**{**h_params.__dict__, 'S0': h_params.S0 + 1.0})
h_dn = HestonParams(**{**h_params.__dict__, 'S0': h_params.S0 - 1.0})
S_up, *_ = simulate_heston(h_up, n_heston, n_steps_h, np.random.default_rng(seed))
S_dn, *_ = simulate_heston(h_dn, n_heston, n_steps_h, np.random.default_rng(seed))
fd_delta_h = disc * (european_call(S_up, K_ATM) - european_call(S_dn, K_ATM)) / 2.0
fd_est = fd_delta_h.mean()
fd_se  = fd_delta_h.std() / np.sqrt(n_heston)

print('Heston Model Delta (ATM Call, N=200,000):')
print(f'  Malliavin BEL : {delta_est:.5f}  ±{delta_se:.5f}  (var = {delta_samples.var():.4f})')
print(f'  FD Central    : {fd_est:.5f}  ±{fd_se:.5f}  (var = {fd_delta_h.var():.4f})')

print('\n(Note: Heston has no simple closed form for delta; both are Monte Carlo estimates)')

## Summary

| Greek | Estimator | Weight formula | Works for discontinuous payoffs? |
|-------|-----------|----------------|----------------------------------|
| Delta | Malliavin/LR | $W_T/(\sigma S_0 T)$ | Yes |
| Delta | Pathwise | $\mathbf{1}_{S_T>K}\cdot S_T/S_0$ | No (requires $f'$) |
| Delta | FD Central | $[V(S_0+h)-V(S_0-h)]/(2h)$ | Yes, but biased for small $h$ |
| Gamma | Malliavin | $(W_T(W_T-\sigma T)-T)/(\sigma^2 T^2 S_0^2)$ | Yes |
| Gamma | FD Central | $[V(S_0+h)-2V(S_0)+V(S_0-h)]/h^2$ | Yes, but 3 sims + noisy |
| Vega  | Malliavin | $(W_T^2-T)/(\sigma T) - W_T$ | Yes |
| Vega  | Pathwise | $\mathbf{1}_{S_T>K}\cdot S_T(W_T-\sigma T)$ | No |

**Key advantages of Malliavin calculus**:
1. Single simulation for all Greeks simultaneously
2. Exact (unbiased) for discontinuous payoffs
3. Extends naturally to path-dependent and multi-asset derivatives
4. Compatible with all variance-reduction techniques